# 01. Exploratory Data Analysis: fraud transactions

**Status:** skeleton. No cell has been run; no findings are recorded here.

**How to use this notebook** (see `LEARNING_LOG.md`):
1. Before running each section, write your *prediction* in the "Prediction" cell.
2. Run it, then fill in "Observation" with what you actually saw.
3. Only then move on. Copy verified facts into `data/README.md`.

Nothing in this notebook assumes the dataset's size, fraud rate, or time span. Column names used
below are *expected* names from the planned Kaggle dataset; section 1 checks them and reports mismatches.

## 0. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

# Adjust after downloading (see data/README.md). Set FILES to None to auto-detect CSVs in data/raw.
DATA_DIR = Path("../data/raw")
FILES = None            # e.g. ["fraudTrain.csv", "fraudTest.csv"]; verify the real file names
SAMPLE_ROWS = None      # e.g. 150_000 for quick iteration; None = load everything
RANDOM_SEED = 42

## 1. Load and verify schema

In [ ]:
csvs = sorted(DATA_DIR.glob("*.csv")) if FILES is None else [DATA_DIR / f for f in FILES]
print("Files found:", [p.name for p in csvs])
assert csvs, f"No CSV files in {DATA_DIR.resolve()}; download the dataset first (data/README.md)."

frames = []
for p in csvs:
    f = pd.read_csv(p, nrows=SAMPLE_ROWS)
    f["source_file"] = p.name
    frames.append(f)
    print(f"{p.name}: {f.shape[0]:,} rows x {f.shape[1]} cols")
df = pd.concat(frames, ignore_index=True)
df.head()

In [ ]:
# Expected columns (from planning notes; NOT yet verified). Edit this mapping to match reality.
EXPECTED = {
    "label": "is_fraud",
    "time": "trans_date_trans_time",
    "customer": "cc_num",
    "merchant": "merchant",
    "category": "category",
    "amount": "amt",
    "cust_lat": "lat", "cust_long": "long",
    "merch_lat": "merch_lat", "merch_long": "merch_long",
    "state": "state",
}
missing = {k: v for k, v in EXPECTED.items() if v not in df.columns}
extra = [c for c in df.columns if c not in EXPECTED.values()]
print("Missing expected columns:", missing or "none")
print("Other columns:", extra)
df.dtypes

**Observation (fill in):** actual columns, dtypes, anything that differs from expectations.
Update the `EXPECTED` mapping and `data/README.md` accordingly.

## 2. Basic shape, quality and duplicates

**Questions to answer:**

- How many rows and columns in total, and per source file?
- Missing values per column? Any placeholder values (empty strings, -1, 0)?
- Duplicate rows? Duplicate transaction identifiers (if an id column exists)?
- Do the files overlap in time or customers?

**Prediction (write before running):** _..._

In [ ]:
print(df.shape)
display(df.isna().sum().to_frame("n_missing").assign(pct=lambda x: 100 * x.n_missing / len(df)))
print("Fully duplicated rows:", df.duplicated().sum())
display(df.describe(include="all").T)

**Observation (fill in after running):** _..._

## 3. Target: class balance

**Questions to answer:**

- Exact fraud count and fraud rate overall and per source file?
- Is the rate stable across files and over time? (If not, a time split behaves very differently from a random one.)
- Given the rate, what accuracy would a model that always predicts 'legit' get? Why does that matter?

**Prediction (write before running):** _..._

In [ ]:
y = EXPECTED["label"]
counts = df[y].value_counts(dropna=False)
print(counts)
print("Fraud rate overall: %.4f%%" % (100 * df[y].mean()))
display(df.groupby("source_file")[y].agg(["count", "sum", "mean"]))
print("Accuracy of always predicting 0: %.4f" % (1 - df[y].mean()))

**Observation (fill in after running):** _..._

## 4. Time coverage and ordering

**Questions to answer:**

- What is the date range, and per file?
- Is the data already sorted by time? Are timestamps timezone-aware?
- Fraud rate by month: stable, trending, or bursty?
- Where would a sensible chronological train/validation/test cut go, and how many frauds land in each part?

**Prediction (write before running):** _..._

In [ ]:
t = pd.to_datetime(df[EXPECTED["time"]])
df["_ts"] = t
print("Range:", t.min(), "->", t.max())
print("Sorted by time:", t.is_monotonic_increasing)
display(df.groupby("source_file")["_ts"].agg(["min", "max"]))
monthly = df.set_index("_ts")[EXPECTED["label"]].resample("MS").agg(["count", "sum", "mean"])
display(monthly)
monthly["mean"].plot(title="Fraud rate by month", figsize=(8, 3)); plt.show()

**Observation (fill in after running):** _..._

## 5. Customers and per-user history depth

**Questions to answer:**

- How many unique customers and merchants?
- Transactions per customer: distribution, min, median, max. Is there enough history per user for 'amount vs. user average' and velocity features?
- Does each customer appear in both early and late periods (needed for point-in-time baselines)?
- Are there customers with fraud only once vs. repeatedly?

**Prediction (write before running):** _..._

In [ ]:
c = EXPECTED["customer"]
per_user = df.groupby(c).size()
print("Customers:", per_user.size, "| Merchants:", df[EXPECTED["merchant"]].nunique())
display(per_user.describe())
per_user.hist(bins=50, figsize=(8, 3)); plt.title("Transactions per customer"); plt.show()
fraud_per_user = df.groupby(c)[EXPECTED["label"]].sum()
print("Customers with >=1 fraud:", (fraud_per_user > 0).sum())

**Observation (fill in after running):** _..._

## 6. Amount

**Questions to answer:**

- Distribution of amount (use a log scale): skewed? outliers? zeros or negatives?
- Does the fraud amount distribution differ from legit? By how much, and is the difference robust (compare medians and quantiles, not just means)?
- Are there typical 'card testing' small amounts among frauds?

**Prediction (write before running):** _..._

In [ ]:
a, y = EXPECTED["amount"], EXPECTED["label"]
display(df.groupby(y)[a].describe(percentiles=[.05, .25, .5, .75, .95, .99]))
fig, ax = plt.subplots(figsize=(8, 3))
for k, g in df.groupby(y):
    ax.hist(np.log1p(g[a].clip(lower=0)), bins=60, alpha=.5, density=True, label=f"{y}={k}")
ax.set_xlabel("log1p(amount)"); ax.legend(); plt.show()

**Observation (fill in after running):** _..._

## 7. Time-of-day and day-of-week patterns

**Questions to answer:**

- Fraud rate by hour of day: is there a night effect? How big, and based on how many frauds per bucket?
- By day of week and by month?
- Beware tiny buckets: report counts next to rates.

**Prediction (write before running):** _..._

In [ ]:
y = EXPECTED["label"]
df["_hour"] = df["_ts"].dt.hour
df["_dow"] = df["_ts"].dt.dayofweek
for col in ["_hour", "_dow"]:
    display(df.groupby(col)[y].agg(["count", "sum", "mean"]))

**Observation (fill in after running):** _..._

## 8. Category and merchant

**Questions to answer:**

- Fraud rate by category, with counts. Which categories have enough frauds to trust the rate?
- How many merchants appear only once? (Implications for a merchant-frequency feature and for high-cardinality encoding.)
- Does a merchant's name reveal anything unexpected (e.g. a prefix)? Inspect before using.

**Prediction (write before running):** _..._

In [ ]:
y = EXPECTED["label"]
display(df.groupby(EXPECTED["category"])[y].agg(["count", "sum", "mean"]).sort_values("mean", ascending=False))
m = df[EXPECTED["merchant"]].value_counts()
print("Merchants seen once:", (m == 1).sum(), "of", len(m))
display(m.head(10))

**Observation (fill in after running):** _..._

## 9. Geography

**Questions to answer:**

- Is the data confined to one country? (This decides whether a 'new country' flag is meaningful or only 'new state' is.)
- Distance between customer location and merchant location: distribution for fraud vs. legit.
- Does the customer's home location stay constant across rows? (If it is a static home address, 'distance from previous transaction' must use another location signal; decide and record in an ADR.)

**Prediction (write before running):** _..._

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    d = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(d))

e = EXPECTED
df["_dist_km"] = haversine_km(df[e["cust_lat"]], df[e["cust_long"]], df[e["merch_lat"]], df[e["merch_long"]])
display(df.groupby(e["label"])["_dist_km"].describe())
print("States:", df[e["state"]].nunique())
print("Customers with >1 distinct home coordinate:",
      (df.groupby(e["customer"])[[e["cust_lat"], e["cust_long"]]].nunique().max(axis=1) > 1).sum())

**Observation (fill in after running):** _..._

## 10. Velocity prototype (exploration only, not the production feature)

**Questions to answer:**

- Within one customer, how close in time are consecutive transactions? Distribution of gaps.
- Do frauds cluster in short bursts per customer? Compare gap-to-previous-transaction for fraud vs. legit.
- Whatever you compute here uses only *past* rows per customer. Convince yourself why that matters, then write the real version in `src/features/` with tests.

**Prediction (write before running):** _..._

In [ ]:
d = df.sort_values([EXPECTED["customer"], "_ts"]).copy()
d["_gap_s"] = d.groupby(EXPECTED["customer"])["_ts"].diff().dt.total_seconds()
display(d.groupby(EXPECTED["label"])["_gap_s"].describe(percentiles=[.05, .25, .5, .75]))

**Observation (fill in after running):** _..._

## 11. Leakage and suitability checks

**Questions to answer:**

- Which columns must be dropped as identifiers or leaks (names, addresses, transaction ids, card numbers as raw features)?
- Does any column look suspiciously predictive on its own? Investigate rather than celebrate.
- Is the dataset synthetic? What does that imply about which patterns are real vs. artefacts of the generator? (State this limitation in the README results later.)

**Prediction (write before running):** _..._

In [ ]:
display(df.nunique().sort_values(ascending=False).to_frame("n_unique").assign(dtype=df.dtypes.astype(str)))

**Observation (fill in after running):** _..._

## 12. Summary of verified facts (fill in last)

| Fact | Value |
|---|---|
| Files | |
| Rows / columns | |
| Time span | |
| Fraud rate | |
| Customers / merchants | |
| Chosen split dates | |
| Columns dropped and why | |

**Decisions to record as ADRs:** split strategy, primary metric, handling of geography features, columns excluded.

**Update next:** `data/README.md` (verified facts), `LEARNING_LOG.md`.